# NLP — โจทย์แบบ "จำแนกข้อความ" (Text Classification)

ข้อความ 1 ชิ้น -> 1 ป้าย (เช่น รีวิว บวก/ลบ/กลาง, หัวข้อข่าว, สแปม/ไม่สแปม)

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = ตัดคำไทย + `TF-IDF` + `Logistic Regression` (เร็ว ไม่ต้อง GPU)
- วิธีที่ 2 (ไม่บังคับ ต้อง GPU) = `WangchanBERTa` คะแนนสูงกว่า


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ input เป็นข้อความ และตอบ "1 ป้ายต่อทั้งข้อความ"
ถ้าต้องป้ายทีละคำ/ตัวอักษร (ตัดคำ/NER) -> ใช้ `thai_word_segmentation.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, คอลัมน์ข้อความ `TEXT_COL`, คอลัมน์ป้าย `LABEL_COL`

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install pythainlp scikit-learn pandas numpy
!pip -q install transformers datasets torch   # วิธีที่ 2 เท่านั้น

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "ใส่-slug-ของ-competition-text-ตรงนี้"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV  = os.path.join(DATA_DIR,"train.csv")   # << แก้ชื่อไฟล์
TEST_CSV   = os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
TEXT_COL   = "text"      # << แก้: คอลัมน์ข้อความ
LABEL_COL  = "label"     # << แก้: คอลัมน์ป้าย
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
ID_COL=sample.columns[0]; ANS_COL=sample.columns[1]
print("train คอลัมน์:",list(train.columns)); display(train.head()); display(sample.head())

## วิธีที่ 1 — ตัดคำ + TF-IDF + Logistic Regression (ง่ายสุด เร็ว ไม่ต้อง GPU)

หลักคิด: เปลี่ยนข้อความเป็นตัวเลข (นับคำสำคัญด้วย TF-IDF) แล้วให้โมเดลเส้นตรงเรียนรู้
ตัดคำไทยก่อนด้วย PyThaiNLP เพราะไทยไม่มีเว้นวรรค

In [ ]:
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

def th_tokenize(s): return word_tokenize(str(s), engine="newmm")   # ตัดคำไทย
clf = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=th_tokenize, ngram_range=(1,2), min_df=2)),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
# เช็คคะแนนในเครื่องก่อน (cross-validation)
print("CV accuracy:", cross_val_score(clf, train[TEXT_COL].astype(str), train[LABEL_COL], cv=5).mean().round(4))
clf.fit(train[TEXT_COL].astype(str), train[LABEL_COL])
out = sample.copy(); out[ANS_COL] = clf.predict(test[TEXT_COL].astype(str))
out.to_csv("submission.csv", index=False)
print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — WangchanBERTa (ไม่บังคับ ต้อง GPU คะแนนสูงกว่า)

ใช้โมเดลภาษาไทยแบบ fine-tune ทำ sequence classification

In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding)
MODEL="airesearch/wangchanberta-base-att-spm-uncased"   # << แก้โมเดลได้
labels=sorted(train[LABEL_COL].unique()); l2i={l:i for i,l in enumerate(labels)}; i2l={i:l for l,i in l2i.items()}
tok=AutoTokenizer.from_pretrained(MODEL)
ds=Dataset.from_dict({"text":train[TEXT_COL].astype(str).tolist(),
                      "label":[l2i[v] for v in train[LABEL_COL]]}).train_test_split(0.1,seed=42)
def enc(b): return tok(b["text"], truncation=True, max_length=256)
ds=ds.map(enc, batched=True)
model=AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))
# หมายเหตุ: ถ้า transformers เก่า (ก่อน 4.46) เปลี่ยน eval_strategy -> evaluation_strategy
args=TrainingArguments(output_dir="out", learning_rate=2e-5, per_device_train_batch_size=16,
    num_train_epochs=3, eval_strategy="epoch", save_strategy="no",
    fp16=torch.cuda.is_available(), report_to="none")
tr=Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
           tokenizer=tok, data_collator=DataCollatorWithPadding(tok))
tr.train()
import torch
preds=[]
model.eval()
for i in range(0,len(test),64):
    batch=tok(test[TEXT_COL].astype(str).tolist()[i:i+64], truncation=True, max_length=256,
              padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad(): preds+=model(**batch).logits.argmax(1).cpu().tolist()
out=sample.copy(); out[ANS_COL]=[i2l[p] for p in preds]
out.to_csv("submission_bert.csv",index=False); print("บันทึก submission_bert.csv")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "tfidf logreg text classification"
!kaggle competitions submissions -c {COMP} | head